In [1]:
import os
import re
import glob
import pickle
import shutil
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Use existing code-folder function when available
try:
    from AnalasysFunction import BurstC as _BurstC
except Exception:
    _BurstC = None

VOL_SR = 500.0
SIMPLE_ISI_MS = 30.0
FORCE_SIMPLE_BURST_GROUPING = True
TAIL_RATIO_THR = 0.30   # exclude later event when (prev_tail_min / current_event_max) >= this threshold
ISO_PRE_MS = 100.0
ISO_POST_MS = 60.0
MIN_UNDETECTABLE_PEAK = 0.05  # if peak dF/F below this, include event directly
MAX_PEAK_SEARCH_S = 0.150  # peak search within 150 ms from event start
CS_Z_START_THR = 1.5
CS_Z_END_THR = 0.5
EVENT_END_FRAC = 0.20
PRE_BASELINE_S = 0.5


def _as_sorted_unique_int(x):
    if x is None:
        return np.array([], dtype=int)
    a = np.asarray(x, dtype=int).ravel()
    if a.size == 0:
        return np.array([], dtype=int)
    return np.unique(a)
def _safe_cal_sr(row_val, default=30.0):
    try:
        v = float(row_val)
        if np.isfinite(v) and v > 0:
            return v
    except Exception:
        pass
    return float(default)
def _zscore_low8(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan
    n = max(1, int(round(0.08 * x.size)))
    lo = np.sort(x)[:n]
    return float(np.mean(lo)), float(np.std(lo))
def _suffix_from_pkl_name(path):
    name = os.path.basename(path)
    for pat in (r"final_correct_spike_detection(.*?)\.pkl$", r"spike_detection_refined_new(.*?)\.pkl$"):
        m = re.search(pat, name, flags=re.IGNORECASE)
        if m:
            s = m.group(1)
            return "main" if s == "" else s
    return os.path.splitext(name)[0]

def _natural_pkl_key(path):
    name = os.path.basename(path)
    m = re.search(r"(final_correct_spike_detection|spike_detection_refined_new)(.*?)\.pkl$", name, flags=re.IGNORECASE)
    if not m:
        return (2, 1, name.lower())

    prefix, suffix = m.group(1).lower(), m.group(2)
    pref_rank = 0 if prefix.startswith("final_correct") else 1

    if suffix == "":
        return (pref_rank, 0, -1)
    try:
        return (pref_rank, 0, int(suffix))
    except Exception:
        return (pref_rank, 1, suffix.lower())
def _find_spike_pkls(cell_folder):
    final_pkls = glob.glob(os.path.join(cell_folder, "final_correct_spike_detection*.pkl"))
    if final_pkls:
        return sorted(list(dict.fromkeys(final_pkls)), key=_natural_pkl_key)
    refined_pkls = glob.glob(os.path.join(cell_folder, "spike_detection_refined_new*.pkl"))
    return sorted(list(dict.fromkeys(refined_pkls)), key=_natural_pkl_key)

def _interp_nan_1d(x):
    x = np.asarray(x, float)
    n = x.size
    if n == 0:
        return x
    idx = np.arange(n)
    good = np.isfinite(x)
    if not np.any(good):
        return np.zeros_like(x)
    if np.sum(good) == 1:
        y = np.full_like(x, x[good][0])
        return y
    y = x.copy()
    y[~good] = np.interp(idx[~good], idx[good], x[good])
    return y

def _moving_average_1d(x, w):
    x = np.asarray(x, float)
    w = max(1, int(w))
    if x.size == 0 or w <= 1:
        return x.copy()
    ker = np.ones(w, dtype=float) / float(w)
    return np.convolve(x, ker, mode="same")

def _build_subthreshold_z(trace_vol, spike_idx, vol_sr=500.0, remove_radius=3, smooth_ms=20.0):
    v = np.asarray(trace_vol, float).copy()
    n = v.size
    if n == 0:
        return v

    sidx = _as_sorted_unique_int(spike_idx)
    sidx = sidx[(sidx >= 0) & (sidx < n)]
    for s in sidx:
        a = max(0, int(s) - int(remove_radius))
        b = min(n, int(s) + int(remove_radius) + 1)
        v[a:b] = np.nan

    v = _interp_nan_1d(v)
    w = max(1, int(round((float(smooth_ms) / 1000.0) * float(vol_sr))))
    v_lp = _moving_average_1d(v, w)

    mu, sd = _zscore_low8(v_lp)
    if not np.isfinite(sd) or sd <= 0:
        return np.zeros_like(v_lp)
    return (v_lp - float(mu)) / float(sd)

def _cs_bounds_from_subz(subz, first_sp, last_sp, trace_len, z_start_thr=1.5, z_end_thr=0.5, vol_sr=500.0,
                         pre_ms=150.0, post_ms=200.0):
    n = int(trace_len)
    if n <= 0:
        return 0, 0

    fs = int(max(0, min(n - 1, int(first_sp))))
    ls = int(max(fs, min(n - 1, int(last_sp))))

    pad_pre = int(round((float(pre_ms) / 1000.0) * float(vol_sr)))
    pad_post = int(round((float(post_ms) / 1000.0) * float(vol_sr)))

    a = max(0, fs - pad_pre)
    b = min(n - 1, ls + pad_post)

    z = np.asarray(subz, float)
    if z.size != n or b < a:
        return max(0, fs - 5), min(n - 1, ls + 5)

    seg = z[a:b + 1]
    if seg.size == 0 or not np.any(np.isfinite(seg)):
        return max(0, fs - 5), min(n - 1, ls + 5)

    i_peak = int(a + np.nanargmax(seg))
    if not np.isfinite(z[i_peak]) or z[i_peak] < float(z_start_thr):
        return max(0, fs - 5), min(n - 1, ls + 5)

    i = i_peak
    while i > a and np.isfinite(z[i]) and z[i] >= float(z_start_thr):
        i -= 1
    s = int(i + 1 if (np.isfinite(z[i]) and z[i] < float(z_start_thr)) else a)

    j = i_peak
    while j < b and np.isfinite(z[j]) and z[j] >= float(z_end_thr):
        j += 1
    e = int(j - 1 if (np.isfinite(z[j]) and z[j] < float(z_end_thr)) else b)

    # Keep bounds consistent with spike span
    s = min(s, fs)
    e = max(e, ls)

    s = int(max(0, min(n - 1, s)))
    e = int(max(s, min(n - 1, e)))
    return s, e

def _vol_idx_to_cal_idx(v_idx, vol_sr, cal_sr, cal_len):
    if cal_len <= 0:
        return 0
    t = float(v_idx) / float(vol_sr)
    c = int(round(t * float(cal_sr)))
    return int(max(0, min(int(cal_len) - 1, c)))

def _ensure_main_overlay_for_cell(cell_folder, out_infos):
    target_html = os.path.join(cell_folder, "event_spike_overlay_main.html")
    target_svg = os.path.join(cell_folder, "event_spike_overlay_main.svg")

    if len(out_infos) == 0:
        return target_html, target_svg, None

    best = None
    for info in out_infos:
        if _suffix_from_pkl_name(str(info.get("pkl_path", ""))) == "main":
            best = info
            break
    if best is None:
        best = sorted(out_infos, key=lambda x: _natural_pkl_key(str(x.get("pkl_path", ""))))[0]

    src_html = str(best.get("html_path", ""))
    src_svg = str(best.get("svg_path", ""))

    if src_html and os.path.exists(src_html) and os.path.abspath(src_html) != os.path.abspath(target_html):
        shutil.copy2(src_html, target_html)
    if src_svg and os.path.exists(src_svg) and os.path.abspath(src_svg) != os.path.abspath(target_svg):
        shutil.copy2(src_svg, target_svg)

    return target_html, target_svg, src_html

def _save_placeholder_main_overlay(cell_folder, note="No valid events/spikes for overlay"):
    fig = go.Figure()
    fig.add_annotation(text=note, x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False)
    fig.update_layout(template="simple_white", width=1200, height=500,
                      title=f"{os.path.basename(cell_folder)} | event_spike_overlay_main")

    target_html = os.path.join(cell_folder, "event_spike_overlay_main.html")
    target_svg = os.path.join(cell_folder, "event_spike_overlay_main.svg")
    fig.write_html(target_html)
    try:
        fig.write_image(target_svg)
    except Exception as e:
        print(f"[WARN] Placeholder SVG export failed ({target_svg}): {e}")

    return target_html, target_svg

def _group_events_by_isi(spikes, isi_ms=30.0, fs=500.0):
    spikes = _as_sorted_unique_int(spikes)
    if spikes.size == 0:
        return []
    threshold_frames = max(1, int(round((isi_ms / 1000.0) * fs)))

    if _BurstC is not None:
        grouped, _ = _BurstC(spikes.tolist(), threshold_frames)
        return [np.asarray(g, dtype=int) for g in grouped if len(g) > 0]

    events = [[int(spikes[0])]]
    for s in spikes[1:]:
        if (s - events[-1][-1]) <= threshold_frames:
            events[-1].append(int(s))
        else:
            events.append([int(s)])
    return [np.asarray(g, dtype=int) for g in events]

def _events_from_saved_labels(d, trace_len, trace_vol=None, vol_sr=500.0, cs_z_start_thr=1.5, cs_z_end_thr=0.5, non_cs_pad_frames=5, simple_isi_ms=30.0, force_simple_grouping=False):
    """
    Build event list from saved spike classes:
      - simple events regrouped by ISI threshold (default <=30 ms)
        when force_simple_grouping=True, grouping uses vm_all_spikes
      - complex events from `vm_burst_dict` (starts/ends)

    Then compute voltage-domain event bounds:
      - complex: z-score crossing (thr=1.5) on subthreshold depolarization bump
      - non-complex: first_spike-5 ... last_spike+5 frames
    """
    trace_len = int(trace_len)
    if trace_len <= 0:
        return [], np.array([], dtype=int), np.array([], dtype=int)

    simple_spikes = _as_sorted_unique_int(d.get("vm_simple_spikes", []))
    complex_spikes = _as_sorted_unique_int(d.get("vm_complex_spikes", []))
    all_spikes = _as_sorted_unique_int(d.get("vm_all_spikes", []))

    simple_spikes = simple_spikes[(simple_spikes >= 0) & (simple_spikes < trace_len)]
    complex_spikes = complex_spikes[(complex_spikes >= 0) & (complex_spikes < trace_len)]
    all_spikes = all_spikes[(all_spikes >= 0) & (all_spikes < trace_len)]
    if all_spikes.size == 0:
        all_spikes = _as_sorted_unique_int(np.r_[simple_spikes, complex_spikes])

    events = []

    # Complex bursts from vm_burst_dict
    bd = d.get("vm_burst_dict", {})
    starts = np.asarray(bd.get("starts", []), dtype=int).ravel() if isinstance(bd, dict) else np.array([], dtype=int)
    ends = np.asarray(bd.get("ends", []), dtype=int).ravel() if isinstance(bd, dict) else np.array([], dtype=int)
    locs = np.asarray((bd.get("locs", []) if isinstance(bd, dict) else []), dtype=int).ravel()
    if locs.size == 0 and isinstance(bd, dict):
        locs = np.asarray(bd.get("complex_bursts", []), dtype=int).ravel()

    n_complex = min(starts.size, ends.size)
    if n_complex == 0 and locs.size > 0:
        starts = locs.copy()
        ends = locs.copy()
        n_complex = locs.size

    complex_ranges = []
    complex_event_index = {}
    for i in range(n_complex):
        s = int(max(0, min(trace_len - 1, int(starts[i]))))
        e = int(max(s, min(trace_len - 1, int(ends[i]))))
        sp = complex_spikes[(complex_spikes >= s) & (complex_spikes <= e)]
        if sp.size == 0:
            if i < locs.size:
                c = int(locs[i])
                if 0 <= c < trace_len:
                    sp = np.array([c], dtype=int)
            if sp.size == 0:
                sp = np.array([s], dtype=int)

        sp = _as_sorted_unique_int(sp)
        if sp.size == 0:
            continue

        events.append({
            "spikes": sp,
            "n_spikes": int(sp.size),
            "event_type": "complex",
            "event_kind": "complex",
            "start_frame": int(s),
            "end_frame": int(e),
            "pkl_start_frame": int(s),
            "pkl_end_frame": int(e),
            "source": "vm_burst_dict",
        })
        complex_ranges.append((s, e))
        complex_event_index[(int(s), int(e))] = len(events) - 1

    def _overlaps_complex(a, b):
        for cs, ce in complex_ranges:
            if not (b < cs or a > ce):
                return True
        return False

    # Simple events: regroup by ISI threshold.
    # Forced mode uses vm_all_spikes directly so burst grouping is driven only by ISI.
    if bool(force_simple_grouping):
        simple_group_input = all_spikes
        simple_source_tag = "all_spikes_isi_grouped_forced"
    else:
        simple_group_input = np.array(
            [int(s) for s in simple_spikes.tolist() if not _overlaps_complex(int(s), int(s))],
            dtype=int,
        )
        simple_source_tag = "simple_spikes_isi_grouped"

    complex_spike_set = set(int(x) for x in complex_spikes.tolist())
    grouped_simple = _group_events_by_isi(simple_group_input, isi_ms=float(simple_isi_ms), fs=vol_sr)
    for sp in grouped_simple:
        sp = _as_sorted_unique_int(sp)
        if sp.size == 0:
            continue
        has_complex_spike = any((int(x) in complex_spike_set) for x in sp.tolist())
        if has_complex_spike and len(complex_ranges) > 0:
            # Hierarchy: if grouped event contains complex spikes, merge it into
            # the overlapping PKL complex event and keep PKL start/end bounds.
            sp_list = [int(x) for x in sp.tolist()]
            sp_min = int(np.min(sp_list))
            sp_max = int(np.max(sp_list))
            best = None
            best_score = -1
            for cs, ce in complex_ranges:
                if sp_max < cs or sp_min > ce:
                    continue
                ov = sum((cs <= x <= ce) for x in sp_list)
                if ov > best_score:
                    best_score = ov
                    best = (int(cs), int(ce))
            if best is not None and best in complex_event_index:
                idx = int(complex_event_index[best])
                cur_sp = _as_sorted_unique_int(events[idx].get("spikes", []))
                merged_sp = _as_sorted_unique_int(np.r_[cur_sp, sp])
                events[idx]["spikes"] = merged_sp
                events[idx]["n_spikes"] = int(merged_sp.size)
                events[idx]["source"] = str(events[idx].get("source", "vm_burst_dict")) + "|merged_from_isi"
                continue

        ev_type = "simple"
        ev_kind = "single" if int(sp.size) == 1 else "simple_burst"
        ev_source = str(simple_source_tag)
        events.append({
            "spikes": sp,
            "n_spikes": int(sp.size),
            "event_type": ev_type,
            "event_kind": ev_kind,
            "start_frame": int(sp[0]),
            "end_frame": int(sp[-1]),
            "source": ev_source,
        })

    # deterministic order and deduplicate by spikes tuple (complex takes precedence)
    events = sorted(
        events,
        key=lambda x: (
            int(x.get("start_frame", 0)),
            0 if x.get("event_type") == "complex" else 1,
            int(x.get("end_frame", 0)),
            int(x.get("n_spikes", 0)),
        )
    )

    uniq = []
    seen_spikes = set()
    for ev in events:
        sp = _as_sorted_unique_int(ev.get("spikes", []))
        if sp.size == 0:
            continue
        sig = tuple(sp.tolist())
        if sig in seen_spikes:
            continue
        ev["spikes"] = sp
        ev["n_spikes"] = int(sp.size)
        seen_spikes.add(sig)
        uniq.append(ev)

    # Voltage-based event bounds
    if trace_vol is not None and len(uniq) > 0:
        all_sp = _as_sorted_unique_int(d.get("vm_all_spikes", []))
        if all_sp.size == 0:
            all_sp = _as_sorted_unique_int(np.r_[simple_spikes, complex_spikes])
        subz = _build_subthreshold_z(trace_vol, all_sp, vol_sr=vol_sr, remove_radius=3, smooth_ms=20.0)

        pad = int(max(1, non_cs_pad_frames))
        for ev in uniq:
            sp = _as_sorted_unique_int(ev.get("spikes", []))
            if sp.size == 0:
                continue
            fs = int(sp[0])
            ls = int(sp[-1])
            nsp = int(sp.size)
            et = str(ev.get("event_type", "simple"))

            # Requested rule:
            # - complex events: z-threshold crossing bounds
            # - simple bursts with n_spikes > 2: z-threshold crossing bounds
            # - single/small simple events (n_spikes <= 2): +/-5 frames
            use_z_bounds = (et == "complex") or ((et == "simple") and (nsp > 2))

            if use_z_bounds:
                s, e = _cs_bounds_from_subz(subz, fs, ls, trace_len, z_start_thr=cs_z_start_thr, z_end_thr=cs_z_end_thr, vol_sr=vol_sr)
                if et == "complex":
                    ev["bound_method"] = "cs_subthreshold_z1p5"
                else:
                    ev["bound_method"] = "simple_burst_gt2_subthreshold_z1p5"
            else:
                s = max(0, fs - pad)
                e = min(trace_len - 1, ls + pad)
                ev["bound_method"] = "simple_n_le2_pad5"

            ev["start_frame"] = int(s)
            ev["end_frame"] = int(e)

    return uniq, simple_spikes, complex_spikes

def _nearest_idx(t_axis, t):
    if t_axis.size == 0:
        return 0
    return int(np.argmin(np.abs(t_axis - t)))

def _calc_event_features(cal_s, cal_t, start_idx, end_idx, next_start_idx=None,
                         mu_low8=np.nan, sd_low8=np.nan,
                         max_peak_search_s=0.150, pre_baseline_s=0.5, end_frac=0.20,
                         use_baseline_frac_end=False):
    cal_s = np.asarray(cal_s, float)
    cal_t = np.asarray(cal_t, float)
    n = cal_s.size
    if n == 0:
        return None

    dt = float(np.nanmedian(np.diff(cal_t))) if cal_t.size >= 2 else (1.0 / 30.0)
    cal_sr = (1.0 / dt) if dt > 0 else 30.0

    i_start = int(max(0, min(n - 1, int(start_idx))))
    i_end = int(max(i_start, min(n - 1, int(end_idx))))

    if next_start_idx is None:
        i_next_start = n - 1
        i_next_cap = n - 1
    else:
        i_next_start = int(max(i_start, min(n - 1, int(next_start_idx))))
        i_next_cap = int(max(i_start, min(n - 1, i_next_start - 1)))
    i_end = int(max(i_start, min(i_end, i_next_cap)))

    pre_n = max(1, int(round(pre_baseline_s * cal_sr)))
    b0 = max(0, i_start - pre_n)
    b1 = i_start
    if b1 > b0:
        baseline = float(np.nanmedian(cal_s[b0:b1]))
    else:
        baseline = float(mu_low8) if np.isfinite(mu_low8) else float(np.nanmedian(cal_s))

    if (max_peak_search_s is None) or (not np.isfinite(max_peak_search_s)) or (float(max_peak_search_s) <= 0):
        peak_lim = int(min(i_end, max(i_start, i_next_start - 1)))
    else:
        peak_n = max(1, int(round(float(max_peak_search_s) * cal_sr)))
        peak_lim = int(min(i_end, i_start + peak_n, max(i_start, i_next_start - 1)))
    if peak_lim < i_start:
        peak_lim = i_start

    seg = cal_s[i_start:peak_lim + 1]
    clear_peak = False
    if seg.size > 0 and np.any(np.isfinite(seg)):
        rel_peak = int(np.nanargmax(seg))
        i_peak = i_start + rel_peak
        peak_raw = float(cal_s[i_peak])
        peak_df_f = float(peak_raw - baseline)
        clear_peak = (i_peak > i_start) and np.isfinite(peak_df_f) and (peak_df_f > 0)
    else:
        i_peak = i_start
        peak_raw = baseline
        peak_df_f = 0.0

    if not clear_peak:
        i_peak = i_start
        peak_raw = baseline
        peak_df_f = 0.0
        peak_z = 0.0
        hwhm_s = np.nan
        auc = 0.0
        return {
            "start_idx": int(i_start),
            "peak_idx": int(i_peak),
            "end_idx": int(i_end),
            "baseline": float(baseline),
            "peak_raw": float(peak_raw),
            "peak_df_f": float(peak_df_f),
            "peak_z": float(peak_z),
            "hwhm_s": float(hwhm_s) if np.isfinite(hwhm_s) else np.nan,
            "auc": float(auc),
            "clear_peak": False,
        }

    peak_z = float((peak_raw - mu_low8) / sd_low8) if np.isfinite(sd_low8) and sd_low8 > 0 else np.nan

    # Burst calcium end rule:
    # end at first return to baseline + end_frac*(peak-baseline),
    # or at next event start - 1 (whichever comes first).
    if use_baseline_frac_end:
        end_level = float(baseline + float(end_frac) * peak_df_f)
        i_end_upper = int(max(i_peak, i_next_cap))
        right_decay = cal_s[i_peak:i_end_upper + 1]
        if right_decay.size > 0 and np.any(np.isfinite(right_decay)):
            decay_cross = np.where(right_decay <= end_level)[0]
            if decay_cross.size > 0:
                i_end = int(i_peak + int(decay_cross[0]))
            else:
                i_end = int(i_end_upper)
        else:
            i_end = int(i_end_upper)

    hwhm_s = np.nan
    half_level = baseline + 0.5 * peak_df_f
    left_seg = cal_s[i_start:i_peak + 1]
    right_seg = cal_s[i_peak:i_end + 1]
    left_cross = np.where(left_seg <= half_level)[0]
    right_cross = np.where(right_seg <= half_level)[0]
    if left_cross.size > 0 and right_cross.size > 0:
        i_left = i_start + int(left_cross[-1])
        i_right = i_peak + int(right_cross[0])
        if i_right > i_left:
            hwhm_s = float(cal_t[i_right] - cal_t[i_left])

    if i_end > i_start:
        y = np.clip(cal_s[i_start:i_end + 1] - baseline, 0, None)
        auc = float(np.trapezoid(y, x=cal_t[i_start:i_end + 1]))
    else:
        auc = 0.0

    return {
        "start_idx": int(i_start),
        "peak_idx": int(i_peak),
        "end_idx": int(i_end),
        "baseline": float(baseline),
        "peak_raw": float(peak_raw),
        "peak_df_f": float(peak_df_f),
        "peak_z": float(peak_z),
        "hwhm_s": float(hwhm_s) if np.isfinite(hwhm_s) else np.nan,
        "auc": float(auc),
        "clear_peak": True,
    }

def _evaluate_include_rule(events, complex_spikes, simple_burst_starts,
                           vol_sr=500.0,
                           pre_cs_ms=300.0,
                           post_cs_ms=100.0,
                           pre_sb_ms=200.0,
                           trace_cal=None,
                           ratio_thr=0.30,
                           min_undetectable_peak=0.05,
                           cal_sr=30.0,
                           isolate_pre_ms=100.0,
                           isolate_post_ms=60.0):
    """
    Include rule:
      - Apply tail-ratio criterion ONLY when previous event is complex.
      - Otherwise include only if no event starts within [-isolate_pre_ms, +isolate_post_ms]
        around the current event start.
    """
    cal = np.asarray(trace_cal, dtype=float).ravel() if trace_cal is not None else np.array([], dtype=float)

    events = sorted(events, key=lambda ev: int(ev.get("start_idx", ev.get("start_frame", 0))))
    if len(events) == 0:
        return events

    start_idx_arr = np.array([int(ev.get("start_idx", ev.get("start_frame", 0))) for ev in events], dtype=int)
    pre_n = max(0, int(round((float(isolate_pre_ms) / 1000.0) * float(cal_sr))))
    post_n = max(0, int(round((float(isolate_post_ms) / 1000.0) * float(cal_sr))))

    for i, ev in enumerate(events):
        ev_sp = _as_sorted_unique_int(ev.get("spikes", []))
        ev["tail_ratio"] = np.nan
        ev["tail_min_prev"] = np.nan
        ev["curr_max"] = np.nan
        ev["tail_ratio_mode"] = "prev_min_tail_to_curr_max"

        if ev_sp.size == 0:
            ev["include"] = False
            ev["include_reason"] = "empty_event"
            continue

        s_cur = int(ev.get("start_idx", ev.get("start_frame", int(ev_sp[0]))))
        w0 = int(s_cur - pre_n)
        w1 = int(s_cur + post_n)
        neighbor_mask = (start_idx_arr >= w0) & (start_idx_arr <= w1)
        neighbor_count = int(np.sum(neighbor_mask) - 1)
        isolated = (neighbor_count <= 0)
        ev["neighbor_count_isowin"] = int(max(0, neighbor_count))

        prev_is_complex = False
        if i > 0:
            prev = events[i - 1]
            prev_is_complex = (str(prev.get("event_type", "")) == "complex") or (str(prev.get("event_kind", "")) == "complex")
        ev["prev_is_complex"] = bool(prev_is_complex)

        if not prev_is_complex:
            ev["include"] = bool(isolated)
            ev["include_reason"] = "isolated_no_event_100ms_pre_60ms_post" if isolated else "neighbor_in_100ms_pre_60ms_post"
            continue

        # Previous event is complex -> apply ratio criterion.
        peak_df_f = float(ev.get("peak_df_f", np.nan))
        if np.isfinite(peak_df_f) and peak_df_f < float(min_undetectable_peak):
            ev["include"] = True
            ev["include_reason"] = "peak_lt_0p05_prev_complex"
            continue

        if cal.size == 0:
            ev["include"] = True
            ev["include_reason"] = "no_cal_trace_prev_complex"
            continue

        prev = events[i - 1]
        p_prev = int(prev.get("peak_idx", prev.get("start_idx", 0)))
        e_prev = int(prev.get("end_idx", p_prev))
        e_cur = int(ev.get("end_idx", s_cur))

        p_prev = max(0, min(cal.size - 1, p_prev))
        e_prev = max(p_prev, min(cal.size - 1, e_prev))
        s_cur = max(0, min(cal.size - 1, s_cur))
        e_cur = max(s_cur, min(cal.size - 1, e_cur))

        prev_tail = np.asarray(cal[p_prev:e_prev + 1], dtype=float)
        prev_tail = prev_tail[np.isfinite(prev_tail)]
        prev_tail_min = float(np.nanmin(prev_tail)) if prev_tail.size > 0 else np.nan

        curr_seg = np.asarray(cal[s_cur:e_cur + 1], dtype=float)
        curr_seg = curr_seg[np.isfinite(curr_seg)]
        curr_max = float(np.nanmax(curr_seg)) if curr_seg.size > 0 else np.nan

        ev["tail_min_prev"] = prev_tail_min
        ev["curr_max"] = curr_max

        if (not np.isfinite(prev_tail_min)) or (not np.isfinite(curr_max)) or (curr_max <= 0):
            ev["include"] = True
            ev["include_reason"] = "ratio_input_invalid_prev_complex"
            ev["tail_ratio"] = np.nan
            continue

        ratio = float(prev_tail_min / curr_max)
        ev["tail_ratio"] = ratio

        if ratio >= float(ratio_thr):
            ev["include"] = False
            ev["include_reason"] = "tail_ratio_ge_thr_prev_complex"
        else:
            ev["include"] = True
            ev["include_reason"] = "tail_ratio_lt_thr_prev_complex"

    return events

def _safe_linear_slope(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if x.size < 2:
        return np.nan
    if np.nanstd(x) <= 1e-12:
        return np.nan
    try:
        return float(np.polyfit(x, y, 1)[0])
    except Exception:
        return np.nan

def _sanitize_scale(scale, default=1.0, lo=0.05, hi=20.0):
    try:
        s = float(scale)
    except Exception:
        return float(default)
    if (not np.isfinite(s)) or (s <= 0):
        return float(default)
    return float(min(float(hi), max(float(lo), s)))

def _compute_robust_scaling_from_cs(trace_vol, trace_cal, class_events, event_rows,
                                    vol_sr=500.0, cal_sr=30.0, pre_baseline_s=0.5):
    """
    Trace-level robust dF/F approximation using CS events.
    We use per-CS-event scatter of (F0, dF) and slope(dF vs F0),
    then scale = slope / mean_raw_dF_CS.
    """
    v = np.asarray(trace_vol, float).ravel()
    c = np.asarray(trace_cal, float).ravel()
    nv = v.size
    nc = c.size

    out = {
        "scale_vol": 1.0,
        "scale_cal": 1.0,
        "n_cs_used": 0,
        "slope_vol": np.nan,
        "slope_cal": np.nan,
        "raw_cs_amp_vol": np.nan,
        "raw_cs_amp_cal": np.nan,
    }

    if nv == 0 or nc == 0 or len(event_rows) == 0:
        return out

    # silent masks from all classified event windows
    silent_v = np.ones(nv, dtype=bool)
    silent_c = np.ones(nc, dtype=bool)
    for ev in class_events:
        vs = int(max(0, min(nv - 1, int(ev.get("start_frame", 0)))))
        ve = int(max(vs, min(nv - 1, int(ev.get("end_frame", vs)))))
        silent_v[vs:ve + 1] = False
        cs = _vol_idx_to_cal_idx(vs, vol_sr=vol_sr, cal_sr=cal_sr, cal_len=nc)
        ce = _vol_idx_to_cal_idx(ve, vol_sr=vol_sr, cal_sr=cal_sr, cal_len=nc)
        silent_c[cs:ce + 1] = False

    g_f0_v = float(np.nanmedian(v[silent_v])) if np.any(silent_v) else float(np.nanmedian(v))
    g_f0_c = float(np.nanmedian(c[silent_c])) if np.any(silent_c) else float(np.nanmedian(c))

    pre_nv = max(1, int(round(float(pre_baseline_s) * float(vol_sr))))
    pre_nc = max(1, int(round(float(pre_baseline_s) * float(cal_sr))))

    cs_events = [ev for ev in event_rows if str(ev.get("event_type", "")) == "complex"]

    f0_v, df_v = [], []
    f0_c, df_c = [], []

    for ev in cs_events:
        sp = _as_sorted_unique_int(ev.get("spikes", []))
        if sp.size == 0:
            continue

        first_sp = int(max(0, min(nv - 1, int(sp[0]))))
        b0v = max(0, first_sp - pre_nv)
        b1v = first_sp
        if b1v > b0v and np.any(np.isfinite(v[b0v:b1v])):
            base_v = float(np.nanmedian(v[b0v:b1v]))
        else:
            base_v = g_f0_v

        w0v = max(0, first_sp - 1)
        w1v = min(nv, first_sp + 2)
        if w1v <= w0v or not np.any(np.isfinite(v[w0v:w1v])):
            continue
        amp_v = float(np.nanmean(v[w0v:w1v]) - base_v)

        p_idx = int(max(0, min(nc - 1, int(ev.get("peak_idx", ev.get("start_idx", 0))))))
        s_idx = int(max(0, min(nc - 1, int(ev.get("start_idx", p_idx)))))
        b0c = max(0, s_idx - pre_nc)
        b1c = s_idx
        if b1c > b0c and np.any(np.isfinite(c[b0c:b1c])):
            base_c = float(np.nanmedian(c[b0c:b1c]))
        else:
            base_c = g_f0_c

        w0c = max(0, p_idx - 1)
        w1c = min(nc, p_idx + 2)
        if w1c <= w0c or not np.any(np.isfinite(c[w0c:w1c])):
            continue
        amp_c = float(np.nanmean(c[w0c:w1c]) - base_c)

        f0_v.append(base_v)
        df_v.append(amp_v)
        f0_c.append(base_c)
        df_c.append(amp_c)

    f0_v = np.asarray(f0_v, float)
    df_v = np.asarray(df_v, float)
    f0_c = np.asarray(f0_c, float)
    df_c = np.asarray(df_c, float)

    n_use = int(min(f0_v.size, f0_c.size))
    out["n_cs_used"] = n_use

    slope_v = _safe_linear_slope(f0_v, df_v)
    slope_c = _safe_linear_slope(f0_c, df_c)
    raw_v = float(np.nanmean(df_v)) if df_v.size else np.nan
    raw_c = float(np.nanmean(df_c)) if df_c.size else np.nan

    out["slope_vol"] = slope_v
    out["slope_cal"] = slope_c
    out["raw_cs_amp_vol"] = raw_v
    out["raw_cs_amp_cal"] = raw_c

    sc_v = (slope_v / raw_v) if (np.isfinite(slope_v) and np.isfinite(raw_v) and abs(raw_v) > 1e-12) else np.nan
    sc_c = (slope_c / raw_c) if (np.isfinite(slope_c) and np.isfinite(raw_c) and abs(raw_c) > 1e-12) else np.nan

    out["scale_vol"] = _sanitize_scale(sc_v, default=1.0)
    out["scale_cal"] = _sanitize_scale(sc_c, default=1.0)
    return out

def _analyze_single_pkl(cell_folder, pkl_path, cal_sr,
                        vol_sr=500.0,
                        ratio_thr=0.30):
    with open(pkl_path, "rb") as f:
        d = pickle.load(f)

    trace_vol = np.asarray(d.get("trace_vol", []), dtype=float).ravel()
    trace_cal_raw = np.asarray(d.get("trace_cal", []), dtype=float).ravel()
    if trace_vol.size == 0 or trace_cal_raw.size == 0:
        return pd.DataFrame(), None

    trace_cal = trace_cal_raw.copy()

    class_events, simple_spikes, complex_spikes = _events_from_saved_labels(
        d, trace_vol.size, trace_vol=trace_vol, vol_sr=vol_sr, cs_z_start_thr=CS_Z_START_THR, cs_z_end_thr=CS_Z_END_THR, non_cs_pad_frames=5, simple_isi_ms=SIMPLE_ISI_MS, force_simple_grouping=FORCE_SIMPLE_BURST_GROUPING
    )
    if len(class_events) == 0:
        return pd.DataFrame(), None

    vol_t = np.arange(trace_vol.size, dtype=float) / float(vol_sr)
    cal_t = np.arange(trace_cal.size, dtype=float) / float(cal_sr)

    mu_low8, sd_low8 = _zscore_low8(trace_cal)

    simple_burst_starts = np.array(
        [int(ev.get("start_frame", 0)) for ev in class_events if str(ev.get("event_kind", "")) == "simple_burst"],
        dtype=int,
    )

    event_rows = []
    for i, ev in enumerate(class_events):
        ev_spk = _as_sorted_unique_int(ev.get("spikes", []))
        ev_spk = ev_spk[(ev_spk >= 0) & (ev_spk < trace_vol.size)]
        if ev_spk.size == 0:
            continue

        v_start = int(max(0, min(trace_vol.size - 1, int(ev.get("start_frame", int(ev_spk[0]))))))
        v_end = int(max(v_start, min(trace_vol.size - 1, int(ev.get("end_frame", int(ev_spk[-1]))))))

        c_start = _vol_idx_to_cal_idx(v_start, vol_sr=vol_sr, cal_sr=cal_sr, cal_len=trace_cal.size)

        c_next_start = None
        end_capped_by_next_start = False
        if i < len(class_events) - 1:
            next_ev = class_events[i + 1]
            next_v_start = int(max(0, min(trace_vol.size - 1, int(next_ev.get("start_frame", v_end)))))
            if v_end >= next_v_start:
                v_end = int(max(v_start, next_v_start - 1))
                end_capped_by_next_start = True
            c_next_start = _vol_idx_to_cal_idx(next_v_start, vol_sr=vol_sr, cal_sr=cal_sr, cal_len=trace_cal.size)

        c_end = _vol_idx_to_cal_idx(v_end, vol_sr=vol_sr, cal_sr=cal_sr, cal_len=trace_cal.size)
        if c_next_start is not None:
            c_end_cap = int(max(c_start, min(trace_cal.size - 1, c_next_start - 1)))
            if c_end > c_end_cap:
                c_end = c_end_cap
                end_capped_by_next_start = True

        is_complex_event = (str(ev.get("event_type", "")) == "complex") or (str(ev.get("event_kind", "")) == "complex")
        is_burst_event = (str(ev.get("event_kind", "")) == "simple_burst") or is_complex_event
        peak_window_s = None if is_complex_event else MAX_PEAK_SEARCH_S

        feat = _calc_event_features(
            trace_cal, cal_t,
            start_idx=c_start,
            end_idx=c_end,
            next_start_idx=c_next_start,
            mu_low8=mu_low8,
            sd_low8=sd_low8,
            max_peak_search_s=peak_window_s,
            pre_baseline_s=PRE_BASELINE_S,
            end_frac=EVENT_END_FRAC,
            use_baseline_frac_end=is_burst_event,
        )
        if feat is None:
            continue

        t_start = float(vol_t[v_start])

        event_rows.append({
            "event_idx": len(event_rows),
            "spikes": ev_spk,
            "n_spikes": int(ev_spk.size),
            "event_type": str(ev.get("event_type", "simple")),
            "event_kind": str(ev.get("event_kind", "single")),
            "bound_method": (str(ev.get("bound_method", "")) + ("|next_start_minus1" if end_capped_by_next_start else "")),
            "start_t": t_start,
            "start_frame_v": int(v_start),
            "end_frame_v": int(v_end),
            "is_burst_event": bool(is_burst_event),
            "is_complex_event": bool(is_complex_event),
            "end_capped_by_next_start": bool(end_capped_by_next_start),
            **feat,
        })

    if len(event_rows) == 0:
        return pd.DataFrame(), None

    event_rows = _evaluate_include_rule(
        event_rows,
        complex_spikes=complex_spikes,
        simple_burst_starts=simple_burst_starts,
        vol_sr=vol_sr,
        trace_cal=trace_cal,
        ratio_thr=ratio_thr,
        min_undetectable_peak=MIN_UNDETECTABLE_PEAK,
        cal_sr=cal_sr,
        isolate_pre_ms=ISO_PRE_MS,
        isolate_post_ms=ISO_POST_MS,
    )

    # Additional robust stage (trace-level approximation)
    robust_info = _compute_robust_scaling_from_cs(
        trace_vol=trace_vol,
        trace_cal=trace_cal,
        class_events=class_events,
        event_rows=event_rows,
        vol_sr=vol_sr,
        cal_sr=cal_sr,
        pre_baseline_s=PRE_BASELINE_S,
    )
    robust_scale_vol = float(robust_info.get("scale_vol", 1.0))
    robust_scale_cal = float(robust_info.get("scale_cal", 1.0))

    trace_cal_robust = trace_cal * robust_scale_cal
    robust_mu_low8, robust_sd_low8 = _zscore_low8(trace_cal_robust)

    include_single = []
    include_simple_burst = []
    include_complex = []
    excluded = []
    metric_rows = []

    for ev in event_rows:
        sp = _as_sorted_unique_int(ev["spikes"])
        if ev["include"]:
            kind = str(ev.get("event_kind", "single"))
            if kind == "complex" or ev["event_type"] == "complex":
                include_complex.extend(sp.tolist())
            elif kind == "simple_burst":
                include_simple_burst.extend(sp.tolist())
            else:
                include_single.extend(sp.tolist())

            peak_at_start = int(ev.get("peak_idx", -1)) == int(ev.get("start_idx", -2))
            peak_df_f_out = 0.0 if peak_at_start else float(ev["peak_df_f"])
            peak_z_out = 0.0 if peak_at_start else float(ev["peak_z"])

            robust_feat = _calc_event_features(
                trace_cal_robust,
                cal_t,
                start_idx=int(ev.get("start_idx", 0)),
                end_idx=int(ev.get("end_idx", int(ev.get("start_idx", 0)))),
                next_start_idx=None,
                mu_low8=robust_mu_low8,
                sd_low8=robust_sd_low8,
                max_peak_search_s=(None if bool(ev.get("is_complex_event", False)) else MAX_PEAK_SEARCH_S),
                pre_baseline_s=PRE_BASELINE_S,
                end_frac=EVENT_END_FRAC,
                use_baseline_frac_end=bool(ev.get("is_burst_event", False)),
            )

            if robust_feat is None:
                robust_peak_df_f = np.nan
                robust_peak_z = np.nan
                robust_hwhm_s = np.nan
                robust_auc = np.nan
            else:
                robust_peak_df_f = float(robust_feat["peak_df_f"])
                robust_peak_z = float(robust_feat["peak_z"])
                robust_hwhm_s = float(robust_feat["hwhm_s"]) if np.isfinite(robust_feat["hwhm_s"]) else np.nan
                robust_auc = float(robust_feat["auc"])

            if peak_at_start:
                robust_peak_df_f = 0.0
                robust_peak_z = 0.0

            metric_rows.append({
                "cell_folder": cell_folder,
                "pkl_name": os.path.basename(pkl_path),
                "suffix": _suffix_from_pkl_name(pkl_path),
                "event_idx": int(ev["event_idx"]),
                "event_type": ev["event_type"],
                "n_spikes": int(ev["n_spikes"]),
                "peak_df_f": peak_df_f_out,
                "peak_z": peak_z_out,
                "hwhm_s": float(ev["hwhm_s"]) if np.isfinite(ev["hwhm_s"]) else np.nan,
                "auc": float(ev["auc"]),
                "robust_peak_df_f": robust_peak_df_f,
                "robust_peak_z": robust_peak_z,
                "robust_hwhm_s": robust_hwhm_s,
                "robust_auc": robust_auc,
                "robust_scale_cal": robust_scale_cal,
                "robust_scale_vol": robust_scale_vol,
                "robust_n_cs_used": int(robust_info.get("n_cs_used", 0)),
                "robust_slope_cal": float(robust_info.get("slope_cal", np.nan)),
                "robust_slope_vol": float(robust_info.get("slope_vol", np.nan)),
                "tail_ratio": float(ev["tail_ratio"]) if np.isfinite(ev["tail_ratio"]) else np.nan,
                "bound_method": str(ev.get("bound_method", "")),
                "include_reason": ev["include_reason"],
                "clear_peak": bool(ev.get("clear_peak", False)),
                "start_idx": int(ev.get("start_idx", -1)),
                "peak_idx": int(ev.get("peak_idx", -1)),
                "peak_at_start": bool(peak_at_start),
            })
        else:
            excluded.extend(sp.tolist())

    include_single = _as_sorted_unique_int(include_single)
    include_simple_burst = _as_sorted_unique_int(include_simple_burst)
    include_complex = _as_sorted_unique_int(include_complex)
    excluded = _as_sorted_unique_int(excluded)

    include_single = include_single[(include_single >= 0) & (include_single < trace_vol.size)]
    include_simple_burst = include_simple_burst[(include_simple_burst >= 0) & (include_simple_burst < trace_vol.size)]
    include_complex = include_complex[(include_complex >= 0) & (include_complex < trace_vol.size)]
    excluded = excluded[(excluded >= 0) & (excluded < trace_vol.size)]

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Scatter(x=vol_t, y=trace_vol, mode="lines", name="Voltage trace",
                   line=dict(color="#f4a261", width=1.0)),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(x=cal_t, y=trace_cal, mode="lines", name="Calcium trace",
                   line=dict(color="gray", width=1.0)),
        secondary_y=True,
    )

    if excluded.size:
        fig.add_trace(
            go.Scatter(x=vol_t[excluded], y=trace_vol[excluded], mode="markers", name="Non-chosen events",
                       marker=dict(color="gray", size=6, opacity=0.75)),
            secondary_y=False,
        )
    if include_single.size:
        fig.add_trace(
            go.Scatter(x=vol_t[include_single], y=trace_vol[include_single], mode="markers", name="Chosen single spike",
                       marker=dict(color="#1f77b4", size=7, opacity=0.95)),
            secondary_y=False,
        )
    if include_simple_burst.size:
        fig.add_trace(
            go.Scatter(x=vol_t[include_simple_burst], y=trace_vol[include_simple_burst], mode="markers", name="Chosen simple burst",
                       marker=dict(color="green", size=7, opacity=0.95)),
            secondary_y=False,
        )
    if include_complex.size:
        fig.add_trace(
            go.Scatter(x=vol_t[include_complex], y=trace_vol[include_complex], mode="markers", name="Chosen complex",
                       marker=dict(color="black", size=7, opacity=0.95)),
            secondary_y=False,
        )

    chosen_events = [ev for ev in event_rows if bool(ev.get("include", False))]
    peak_events = [ev for ev in event_rows if int(ev.get("peak_idx", -1)) >= 0 and int(ev.get("peak_idx", -1)) < trace_cal.size]
    chosen_peak_events = [ev for ev in peak_events if bool(ev.get("include", False))]
    nonchosen_peak_events = [ev for ev in peak_events if not bool(ev.get("include", False))]

    if len(chosen_peak_events) > 0:
        p_idx_ch = np.array([int(ev["peak_idx"]) for ev in chosen_peak_events], dtype=int)
        fig.add_trace(
            go.Scatter(x=cal_t[p_idx_ch], y=trace_cal[p_idx_ch], mode="markers", name="Chosen event peak",
                       marker=dict(color="purple", size=11, symbol="star")),
            secondary_y=True,
        )

    if len(nonchosen_peak_events) > 0:
        p_idx_nch = np.array([int(ev["peak_idx"]) for ev in nonchosen_peak_events], dtype=int)
        fig.add_trace(
            go.Scatter(x=cal_t[p_idx_nch], y=trace_cal[p_idx_nch], mode="markers", name="Non-chosen event peak",
                       marker=dict(color="gray", size=11, symbol="star")),
            secondary_y=True,
        )

    # Debug overlay: bound_method + tail_ratio
    if len(event_rows) > 0:
        dbg_events = [ev for ev in event_rows if int(ev.get("peak_idx", -1)) >= 0 and int(ev.get("peak_idx", -1)) < trace_cal.size]
        if len(dbg_events) > 0:
            dbg_idx = np.array([int(ev["peak_idx"]) for ev in dbg_events], dtype=int)
            dbg_text = []
            dbg_label_text = []
            dbg_color = []
            for ev in dbg_events:
                tr = ev.get("tail_ratio", np.nan)
                tr_txt = f"{float(tr):.3f}" if np.isfinite(tr) else "nan"
                bm = str(ev.get("bound_method", ""))
                rsn = str(ev.get("include_reason", ""))
                inc = bool(ev.get("include", False))
                dbg_text.append(
                    f"ev={int(ev.get('event_idx', -1))} | include={inc} | ratio={tr_txt} | bound={bm} | reason={rsn}"
                )
                # Visible short label mainly for excluded events
                if inc:
                    dbg_label_text.append("")
                    dbg_color.append("rgba(0,0,0,0.15)")
                else:
                    dbg_label_text.append(f"r={tr_txt} | {bm}")
                    dbg_color.append("rgba(180,0,0,0.85)")

            fig.add_trace(
                go.Scatter(
                    x=cal_t[dbg_idx],
                    y=trace_cal[dbg_idx],
                    mode="markers+text",
                    name="Event debug",
                    marker=dict(size=8, color=dbg_color, symbol="circle-open"),
                    text=dbg_label_text,
                    textposition="top center",
                    textfont=dict(size=9, color="rgba(120,0,0,0.95)"),
                    customdata=np.array(dbg_text, dtype=object),
                    hovertemplate="%{customdata}<extra></extra>",
                ),
                secondary_y=True,
            )

    suffix = _suffix_from_pkl_name(pkl_path)
    out_base = os.path.join(cell_folder, f"event_spike_overlay_{suffix}")

    fig.update_layout(
        template="simple_white",
        title=(
            f"{os.path.basename(cell_folder)} | {os.path.basename(pkl_path)}"
            f"<br><sup>events from {'vm_all_spikes' if FORCE_SIMPLE_BURST_GROUPING else 'vm_simple_spikes'} regrouped by ISI<={SIMPLE_ISI_MS:.0f}ms + vm_burst_dict | calcium raw (no smoothing) | peak window: simple={MAX_PEAK_SEARCH_S*1000:.0f}ms, complex=full PKL window | "
            f"include rule: ratio only if previous event is complex (thr={TAIL_RATIO_THR:.2f}); otherwise include only if no event in -{ISO_PRE_MS:.0f}ms/+{ISO_POST_MS:.0f}ms window | bounds: start z>=1.5 for CS+simple(n>2); simple(n<=2)=+-5fr; burst end at baseline+{EVENT_END_FRAC:.0%}*rise or next_start-1 | "
            f"robust scales: cal={robust_scale_cal:.3f}, vol={robust_scale_vol:.3f} | debug: hover Event debug for bound/ratio</sup>"
        ),
        width=1400,
        height=650,
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    )
    fig.update_xaxes(title_text="Time (s)")
    fig.update_yaxes(title_text="Voltage", secondary_y=False)
    fig.update_yaxes(title_text="Calcium (dF/F)", secondary_y=True)

    html_path = out_base + ".html"
    svg_path = out_base + ".svg"
    fig.write_html(html_path)
    svg_saved = True
    try:
        fig.write_image(svg_path)
    except Exception as e:
        svg_saved = False
        print(f"[WARN] SVG export failed ({os.path.basename(svg_path)}): {e}")

    print(
        f"[OK] {os.path.basename(pkl_path)} | events={len(event_rows)} | chosen={len(chosen_events)} | "
        f"single={len(include_single)} | simple_burst={len(include_simple_burst)} | complex={len(include_complex)} | "
        f"robust_scales(cal={robust_scale_cal:.3f},vol={robust_scale_vol:.3f},nCS={int(robust_info.get('n_cs_used',0))}) | "
        f"saved_html={os.path.basename(html_path)} | saved_svg={svg_saved}"
    )

    return pd.DataFrame(metric_rows), {
        "figure": fig,
        "html_path": html_path,
        "svg_path": svg_path,
        "cell_folder": cell_folder,
        "pkl_path": pkl_path,
        "suffix": suffix,
    }

def _plot_summary(metrics_df, save_html=None, save_svg=None, show_plot=True,
                  title_prefix="Chosen calcium events vs spike count", metric_prefix=""):
    if metrics_df is None or len(metrics_df) == 0:
        raise ValueError("No chosen events for summary plot.")

    pref = str(metric_prefix or "")
    if pref and not pref.endswith("_"):
        pref = pref + "_"

    n_cells = int(metrics_df["cell_folder"].nunique()) if "cell_folder" in metrics_df.columns else np.nan
    n_complex = int((metrics_df["event_type"] == "complex").sum())
    n_simple = int((metrics_df["event_type"] == "simple").sum())

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Peak (dF/F)", "Peak (z-score)", "HWHM (s)", "AUC"),
        horizontal_spacing=0.10,
        vertical_spacing=0.12,
    )

    panels = [
        (1, 1, f"{pref}peak_df_f", "Peak (dF/F)"),
        (1, 2, f"{pref}peak_z", "Peak (z-score)"),
        (2, 1, f"{pref}hwhm_s", "HWHM (s)"),
        (2, 2, f"{pref}auc", "AUC"),
    ]

    color_map = {"simple": "red", "complex": "black"}
    label_map = {"simple": "Simple/Burst", "complex": "Complex"}

    for r, c, ycol, ytitle in panels:
        if ycol not in metrics_df.columns:
            metrics_df[ycol] = np.nan

        for ev_type in ("simple", "complex"):
            sub = metrics_df[metrics_df["event_type"] == ev_type]
            fig.add_trace(
                go.Scatter(
                    x=sub["n_spikes"],
                    y=sub[ycol],
                    mode="markers",
                    name=label_map[ev_type],
                    marker=dict(color=color_map[ev_type], size=7, opacity=0.8),
                    showlegend=(r == 1 and c == 1),
                ),
                row=r, col=c,
            )
        fig.update_xaxes(title_text="# spikes in event", row=r, col=c)
        fig.update_yaxes(title_text=ytitle, row=r, col=c)

    mode_txt = "robust dF/F" if pref else "raw dF/F"
    fig.update_layout(
        template="simple_white",
        width=1200,
        height=900,
        title=(
            f"{title_prefix}"
            f"<br><sup>mode={mode_txt} | n_cells={n_cells} | n_complex_events={n_complex} | n_simple_events={n_simple}</sup>"
        ),
    )

    if save_html:
        fig.write_html(save_html)
    if save_svg:
        try:
            fig.write_image(save_svg)
        except Exception as e:
            print(f"[WARN] Summary SVG export failed: {e}")

    if show_plot:
        fig.show()
    return fig




In [2]:
# -------------------------------
# Run on all PyrLowFR cells
# -------------------------------
DB_PATH = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv"
SUMMARY_HTML = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary.html"
SUMMARY_SVG = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary.svg"
SUMMARY_ROBUST_HTML = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary_robust.html"
SUMMARY_ROBUST_SVG = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary_robust.svg"

# Set to an integer (e.g., 2) for quick testing, or None for full run
MAX_CELLS = None

DB = pd.read_csv(DB_PATH)
if MAX_CELLS is not None:
    DB = DB.iloc[:int(MAX_CELLS)].copy()

all_metrics = []
all_outputs = []

for row_idx, row in DB.iterrows():
    cell_folder = str(row["Link"])
    cal_sr = _safe_cal_sr(row.get("CALsr", 30.0), default=30.0)

    if not os.path.isdir(cell_folder):
        print(f"[SKIP] missing folder: {cell_folder}")
        continue

    pkl_list = _find_spike_pkls(cell_folder)
    if len(pkl_list) == 0:
        print(f"[SKIP] no spike pkl found: {cell_folder}")
        continue

    print("\n" + "=" * 110)
    print(f"Cell {row_idx + 1}/{len(DB)} | {cell_folder} | CALsr={cal_sr} | pkls={len(pkl_list)}")

    cell_metrics = []
    cell_outputs = []

    for pkl_path in pkl_list:
        try:
            mdf, out_info = _analyze_single_pkl(
                cell_folder=cell_folder,
                pkl_path=pkl_path,
                cal_sr=cal_sr,
                vol_sr=VOL_SR,
                ratio_thr=TAIL_RATIO_THR,
            )

            suffix = _suffix_from_pkl_name(pkl_path)

            if mdf is not None and len(mdf) > 0:
                all_metrics.append(mdf)
                cell_metrics.append(mdf)

                # per-pkl 4-panel summary (requested for multi-pkl cells)
                per_pkl_html = os.path.join(cell_folder, f"event_metrics_4panel_by_{suffix}.html")
                per_pkl_svg = os.path.join(cell_folder, f"event_metrics_4panel_by_{suffix}.svg")
                _plot_summary(
                    mdf,
                    save_html=per_pkl_html,
                    save_svg=per_pkl_svg,
                    show_plot=False,
                    title_prefix=f"{os.path.basename(cell_folder)} | metrics for {os.path.basename(pkl_path)}",
                )
                print(f"[PKL-SUMMARY] saved: {per_pkl_html}")

                per_pkl_html_rb = os.path.join(cell_folder, f"event_metrics_4panel_by_{suffix}_robust.html")
                per_pkl_svg_rb = os.path.join(cell_folder, f"event_metrics_4panel_by_{suffix}_robust.svg")
                _plot_summary(
                    mdf,
                    save_html=per_pkl_html_rb,
                    save_svg=per_pkl_svg_rb,
                    show_plot=False,
                    title_prefix=f"{os.path.basename(cell_folder)} | robust metrics for {os.path.basename(pkl_path)}",
                    metric_prefix="robust_",
                )
                print(f"[PKL-SUMMARY-ROBUST] saved: {per_pkl_html_rb}")

            if out_info is not None:
                all_outputs.append(out_info)
                cell_outputs.append(out_info)

        except Exception as e:
            print(f"[ERROR] {pkl_path}: {e}")

    # Ensure each cell folder has event_spike_overlay_main html+svg
    if len(cell_outputs) > 0:
        main_html, main_svg, src_html = _ensure_main_overlay_for_cell(cell_folder, cell_outputs)
        src_txt = os.path.basename(src_html) if src_html else "n/a"
        print(f"[MAIN-OVERLAY] {os.path.basename(main_html)} + {os.path.basename(main_svg)} (source={src_txt})")
    else:
        main_html, main_svg = _save_placeholder_main_overlay(cell_folder)
        print(f"[MAIN-OVERLAY] placeholder saved: {main_html}")

    # Per-cell (all pkls combined) 4-panel summary
    if len(cell_metrics) > 0:
        cell_df = pd.concat(cell_metrics, ignore_index=True)
        cell_summary_html = os.path.join(cell_folder, "event_metrics_4panel_main.html")
        cell_summary_svg = os.path.join(cell_folder, "event_metrics_4panel_main.svg")
        _plot_summary(
            cell_df,
            save_html=cell_summary_html,
            save_svg=cell_summary_svg,
            show_plot=False,
            title_prefix=f"{os.path.basename(cell_folder)} | chosen calcium events vs spike count",
        )
        print(f"[CELL-SUMMARY] saved: {cell_summary_html}")

        cell_summary_html_rb = os.path.join(cell_folder, "event_metrics_4panel_robust_main.html")
        cell_summary_svg_rb = os.path.join(cell_folder, "event_metrics_4panel_robust_main.svg")
        _plot_summary(
            cell_df,
            save_html=cell_summary_html_rb,
            save_svg=cell_summary_svg_rb,
            show_plot=False,
            title_prefix=f"{os.path.basename(cell_folder)} | robust chosen calcium events vs spike count",
            metric_prefix="robust_",
        )
        print(f"[CELL-SUMMARY-ROBUST] saved: {cell_summary_html_rb}")
    else:
        print(f"[CELL-SUMMARY] skipped (no chosen events): {cell_folder}")

if len(all_metrics) == 0:
    raise RuntimeError("No chosen events found across PyrLowFR cells.")

metrics_df = pd.concat(all_metrics, ignore_index=True)
print(f"\nTotal chosen events: {len(metrics_df)}")
print(metrics_df[["event_type", "n_spikes", "peak_df_f", "peak_z", "hwhm_s", "auc"]].head(10))

max_spk = int(metrics_df["n_spikes"].max()) if len(metrics_df) else 0
n_gt4 = int((metrics_df["n_spikes"] > 4).sum()) if len(metrics_df) else 0
print(f"Max spikes/event (chosen): {max_spk}")
print(f"Chosen events with >4 spikes: {n_gt4}")
print("Chosen-event spike-count distribution:")
print(metrics_df["n_spikes"].value_counts().sort_index())

summary_fig = _plot_summary(metrics_df, save_html=SUMMARY_HTML, save_svg=SUMMARY_SVG, show_plot=True)
summary_fig_rb = _plot_summary(
    metrics_df,
    save_html=SUMMARY_ROBUST_HTML,
    save_svg=SUMMARY_ROBUST_SVG,
    show_plot=True,
    title_prefix="Chosen calcium events vs spike count (robust)",
    metric_prefix="robust_",
)
print(f"Summary saved:\n  HTML: {SUMMARY_HTML}\n  SVG:  {SUMMARY_SVG}\n  Robust HTML: {SUMMARY_ROBUST_HTML}\n  Robust SVG:  {SUMMARY_ROBUST_SVG}")





Cell 1/31 | Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0 | CALsr=30.0 | pkls=1


C:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\scopes\base.py:185: DeprecationWarning:

setDaemon() is deprecated, set the daemon attribute instead



[OK] spike_detection_refined_new.pkl | events=211 | chosen=148 | single=93 | simple_burst=70 | complex=82 | robust_scales(cal=1.000,vol=1.000,nCS=33) | saved_html=event_spike_overlay_main.html | saved_svg=True
[PKL-SUMMARY] saved: Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\event_metrics_4panel_by_main.html
[PKL-SUMMARY-ROBUST] saved: Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\event_metrics_4panel_by_main_robust.html
[MAIN-OVERLAY] event_spike_overlay_main.html + event_spike_overlay_main.svg (source=event_spike_overlay_main.html)
[CELL-SUMMARY] saved: Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\event_metrics_4panel_main.html
[CELL-SUMMARY-ROBUST] saved: Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0\event_metrics_4panel_robust_main.html

Cell 2/31 | Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov2\cell1 | CALsr=30.0 | pkls=1
[OK] spike_detection_refined_new.pkl

Summary saved:
  HTML: Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary.html
  SVG:  Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary.svg
  Robust HTML: Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary_robust.html
  Robust SVG:  Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR_event_metrics_summary_robust.svg
